# 10 - YAMNet Feature Extraction

## Objective

This notebook converts every audio file into a high-level embedding using Google's
pretrained YAMNet model.

These embeddings will later be used to train our own classifier.

Pipeline

Audio
↓

YAMNet

↓

1024-dimensional embedding

↓

Save NumPy arrays

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q tensorflow tensorflow_hub librosa soundfile joblib tqdm

In [3]:
import os
import joblib
import librosa
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub

from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder

In [4]:
SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)

In [5]:
# ============================================================
# CHANGE THESE PATHS
# ============================================================

CSV_FOLDER = "/content/drive/MyDrive/Underwater Audio Data/deep_learning_dataset"

OUTPUT_FOLDER = "/content/drive/MyDrive/Underwater Audio Data/embeddings"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

In [6]:
TRAIN_CSV = os.path.join(CSV_FOLDER, "train.csv")
VALIDATION_CSV = os.path.join(CSV_FOLDER, "validation.csv")
TEST_CSV = os.path.join(CSV_FOLDER, "test.csv")

# Load CSV Files

In [7]:
train_df = pd.read_csv(TRAIN_CSV)

validation_df = pd.read_csv(VALIDATION_CSV)

test_df = pd.read_csv(TEST_CSV)

print(len(train_df))
print(len(validation_df))
print(len(test_df))

3915
406
596


# Label Encoding

Convert

Biological

Vessels

Ambience

↓

0

1

2

In [8]:
encoder = LabelEncoder()

encoder.fit(train_df["class"])

LabelEncoder()

In [9]:
train_df["label"] = encoder.transform(train_df["class"])

validation_df["label"] = encoder.transform(validation_df["class"])

test_df["label"] = encoder.transform(test_df["class"])

In [10]:
joblib.dump(
    encoder,
    os.path.join(
        OUTPUT_FOLDER,
        "label_encoder.pkl"
    )
)

print(encoder.classes_)

['Ambience' 'Biological' 'Vessels']


# Load YAMNet

In [11]:
print("Loading YAMNet...")

yamnet = hub.load("https://tfhub.dev/google/yamnet/1")

print("Loaded successfully.")

Loading YAMNet...
Loaded successfully.


# Audio Loading Functions

This section contains reusable helper functions for loading audio,
resampling, and generating embeddings using YAMNet.

In [12]:
# ============================================================
# Audio Configuration
# ============================================================

TARGET_SAMPLE_RATE = 16000

In [13]:
# ============================================================
# Load Audio
# ============================================================

def load_audio(filepath):

    """
    Load audio as mono waveform at 16 kHz.

    Parameters
    ----------
    filepath : str

    Returns
    -------
    waveform : np.ndarray
    """

    waveform, sr = librosa.load(
        filepath,
        sr=TARGET_SAMPLE_RATE,
        mono=True
    )

    waveform = waveform.astype(np.float32)

    return waveform

In [14]:
# ============================================================
# Test Audio Loading
# ============================================================

sample_audio = train_df.iloc[0]["filepath"]

waveform = load_audio(sample_audio)

print("Shape :", waveform.shape)

print("Duration :", len(waveform)/TARGET_SAMPLE_RATE, "seconds")

Shape : (48000,)
Duration : 3.0 seconds


# YAMNet Embedding Function

In [15]:
# ============================================================
# Extract YAMNet Embedding
# ============================================================

def extract_embedding(filepath):

    """
    Generate a single embedding from an audio file.

    Returns
    -------
    embedding : np.ndarray
        Shape (1024,)
    """

    waveform = load_audio(filepath)

    scores, embeddings, spectrogram = yamnet(waveform)

    embeddings = embeddings.numpy()

    embedding = np.mean(
        embeddings,
        axis=0
    )

    return embedding

In [16]:
embedding = extract_embedding(sample_audio)

print(embedding.shape)

(1024,)


# Batch Feature Extraction

Now we'll convert an entire dataset split into embeddings.

In [17]:
# ============================================================
# Extract Dataset Embeddings
# ============================================================

def process_split(df, split_name):

    embeddings = []

    labels = []

    filepaths = []

    failed_files = []

    print(f"\nProcessing {split_name}")

    for _, row in tqdm(df.iterrows(), total=len(df)):

        filepath = row["filepath"]

        label = row["label"]

        try:

            embedding = extract_embedding(filepath)

            embeddings.append(embedding)

            labels.append(label)

            filepaths.append(filepath)

        except Exception as e:

            failed_files.append(filepath)

    embeddings = np.array(embeddings)

    labels = np.array(labels)

    filepaths = np.array(filepaths)

    print()

    print("Completed")

    print("Embeddings :", embeddings.shape)

    print("Labels :", labels.shape)

    print("Failed :", len(failed_files))

    return embeddings, labels, filepaths, failed_files

In [18]:
train_embeddings, train_labels, train_paths, train_failed = process_split(
    train_df,
    "Training"
)


Processing Training


100%|██████████| 3915/3915 [15:36<00:00,  4.18it/s]


Completed
Embeddings : (3915, 1024)
Labels : (3915,)
Failed : 0


In [19]:
validation_embeddings, validation_labels, validation_paths, validation_failed = process_split(
    validation_df,
    "Validation"
)


Processing Validation


100%|██████████| 406/406 [01:34<00:00,  4.28it/s]


Completed
Embeddings : (406, 1024)
Labels : (406,)
Failed : 0


In [20]:
test_embeddings, test_labels, test_paths, test_failed = process_split(
    test_df,
    "Testing"
)


Processing Testing


100%|██████████| 596/596 [02:23<00:00,  4.15it/s]


Completed
Embeddings : (596, 1024)
Labels : (596,)
Failed : 0


# Verify Shapes

In [21]:
print(train_embeddings.shape)
print(validation_embeddings.shape)
print(test_embeddings.shape)

(3915, 1024)
(406, 1024)
(596, 1024)


# Save Embeddings

In [22]:
# ============================================================
# Save NumPy Arrays
# ============================================================

np.save(os.path.join(OUTPUT_FOLDER, "train_embeddings.npy"), train_embeddings)
np.save(os.path.join(OUTPUT_FOLDER, "train_labels.npy"), train_labels)
np.save(os.path.join(OUTPUT_FOLDER, "train_paths.npy"), train_paths)

np.save(os.path.join(OUTPUT_FOLDER, "validation_embeddings.npy"), validation_embeddings)
np.save(os.path.join(OUTPUT_FOLDER, "validation_labels.npy"), validation_labels)
np.save(os.path.join(OUTPUT_FOLDER, "validation_paths.npy"), validation_paths)

np.save(os.path.join(OUTPUT_FOLDER, "test_embeddings.npy"), test_embeddings)
np.save(os.path.join(OUTPUT_FOLDER, "test_labels.npy"), test_labels)
np.save(os.path.join(OUTPUT_FOLDER, "test_paths.npy"), test_paths)

print("Saved successfully.")

Saved successfully.


In [23]:
print(os.listdir(OUTPUT_FOLDER))

['label_encoder.pkl', 'train_embeddings.npy', 'train_labels.npy', 'train_paths.npy', 'validation_embeddings.npy', 'validation_labels.npy', 'validation_paths.npy', 'test_embeddings.npy', 'test_labels.npy', 'test_paths.npy']
